# 실습 데이터분석가처럼 문제 정의하기

## 서울 공공자전거 대여 데이터

서울시 공공자전거 운영 담당자가 이렇게 말했습니다.

> "자전거가 언제, 어떤 때 많이 쓰이는지 알고 싶습니다."

이 말은 아직 **모호한 문제**입니다. 데이터를 직접 다루며 이 문제를 **분석할 수 있는 질문으로 바꾸고, 질문에 답하는 방법**을 익힙니다.

- 데이터 서울 공공자전거의 1년(365일) × 24시간 대여 기록, 8,760행 14열
- 출처 UCI Machine Learning Repository, Seoul Bike Sharing Demand (https://archive.ics.uci.edu/dataset/560/seoul+bike+sharing+demand)

### 구성

| 번호 | 하는 일 |
|:---|:---|
| 1 | 이론의 EDA 절차를 이 데이터로 그대로 수행합니다. (1.1~1.24) |
| 2 | **문제를 먼저 정의하고**, EDA로 데이터를 점검하며 분석 방법을 설계합니다. (2.1~2.5) |
| 3 | **질문 없이 탐색적 데이터 분석부터 수행하고**, 탐색에서 남은 의문으로 분석 질문을 도출합니다. (3.1~3.7) |

### 진행 방법

- 코드는 **직접** 작성합니다. 각 문항은 알아내야 할 것만 알려 줍니다. 쓸 도구는 스스로 정합니다.
- 코드 셀을 위에서 아래로 차례로 실행합니다. 뒤 문항은 앞 문항의 변수를 사용합니다.
- 코드를 쓰기 전에 **결과를 먼저 예상**합니다. 예상과 다르면 이유를 생각합니다.
- 글로 답하는 문항은 코드 셀 맨 위에 `#` 주석으로 쓰거나, 셀을 추가해 텍스트로 씁니다.
- 숫자만 쓰지 말고 **그 숫자가 무엇을 뜻하는지** 문장으로 씁니다.

### 쓸 수 있는 도구

| 하고 싶은 일 | 도구 |
|:---|:---|
| 데이터 미리보기 | `head()` `tail()` `sample()` |
| 데이터 구조 확인 | `shape` `columns` `dtypes` `info()` |
| Series, 위치 선택 | `pd.Series()` `iloc[ ]` `loc[ ]` `copy()` |
| 변수 선택, 필터링 | `df[["열"]]` `df[조건]` `&` `\|` `unique()` |
| 열 추가, 삭제 | `df["새 열"] = 계산` `drop()` |
| 정렬 | `sort_values()` |
| 결측값, 중복값 확인 | `isna()` `duplicated()` |
| 빈도 계산 | `value_counts()` `len()` |
| 기술통계 | `describe()` `mean()` `median()` |
| 그룹별 집계 | `groupby()` `agg()` `idxmax()` |
| 표 만들기 | `pd.DataFrame()` `round()` |

### 교수용 지도 포인트 (학생용 노트북에는 표시하지 않음)

- **1.20** 대여량 0인 행이 모두 비운영일임을 학생이 스스로 발견하는 것이 핵심입니다. 먼저 알려 주지 마세요.
- **1.22** 기준을 넘는 큰 값이 오류가 아님(여름 저녁의 실제 수요)을 알게 하는 것이 핵심입니다. 이론의 카페 45분과 대비됩니다.
- **3.5** 학생마다 도출하는 질문이 다를 수 있습니다. 어떤 질문이든 다섯 요소와 네 가지 점검(특히 "답이 이미 나와 있지 않은가")을 통과하는지로 확인합니다.

## 준비 데이터 로드

아래 두 셀은 **이미 작성되어 있습니다.** 실행만 하세요. 파일은 `data` 폴더 또는 노트북과 같은 폴더에서 찾습니다. Colab에서는 왼쪽 폴더 아이콘으로 `SeoulBikeData.csv`를 올려 두세요. 이 파일은 온도 기호(°) 때문에 `encoding="latin1"`로 읽어야 글자가 깨지지 않습니다.

In [ ]:
from pathlib import Path
import pandas as pd

candidates = [
    Path("data/SeoulBikeData.csv"),
    Path("강의교재/data/SeoulBikeData.csv"),
    Path("SeoulBikeData.csv"),
]

data_path = next((p for p in candidates if p.exists()), None)
if data_path is None:
    raise FileNotFoundError("SeoulBikeData.csv 파일을 찾을 수 없습니다. 파일을 올린 뒤 다시 실행하세요.")

bike = pd.read_csv(data_path, encoding="latin1")
print("불러온 파일:", data_path)

In [ ]:
# 원본의 긴 열 이름을 짧게 바꿉니다. (원본 파일은 바뀌지 않습니다.)
bike = bike.rename(columns={
    "Date": "date",
    "Rented Bike Count": "rented_count",
    "Hour": "hour",
    "Temperature(°C)": "temperature",
    "Humidity(%)": "humidity",
    "Wind speed (m/s)": "wind_speed",
    "Visibility (10m)": "visibility",
    "Dew point temperature(°C)": "dew_point",
    "Solar Radiation (MJ/m2)": "solar_radiation",
    "Rainfall(mm)": "rainfall",
    "Snowfall (cm)": "snowfall",
    "Seasons": "season",
    "Holiday": "holiday",
    "Functioning Day": "functioning_day",
})
print(bike.columns.tolist())

# 1. EDA 절차 따라 하기

분석 질문이 하나 주어졌습니다. 이 질문에 답하는 과정을 이론과 같은 순서로 밟습니다.

> 서울 공공자전거의 시간대별 평균 대여량은 어떻게 다르며, 어느 시간대에 대여가 가장 많은가?

### 1.1 분석 질문에 필요한 변수 정의

데이터를 열기 전에 먼저 생각합니다. ① 이 질문에 **필요한 변수(열)** ② **필요 없는 변수** ③ **집계 방법**을 쓰세요. 그리고 ④ 분석을 시작하기 전에 데이터가 이 질문에 답할 수 있는지 **점검해야 할 항목 네 가지**를 쓰세요.

**확인값** 필요한 열은 두 개입니다. 네 가지는 이론의 점검표(열 존재, 비어 있음, 값의 범위, 그룹별 건수)를 떠올리세요.

> ① 필요한 변수 `hour`(기준 변수), `rented_count`(측정 변수)
> ② 필요 없는 변수 `temperature`, `season`, `holiday` 등 나머지
> ③ 집계 방법 평균
> ④ 열이 있는가 / 비어 있는 값은 없는가 / 값이 올바른 범위인가 / 시간대마다 건수가 충분한가

### 1.2 변수의 종류 분류

`bike`의 열 14개를 **범주형**과 **연속형**으로 분류하세요. 숫자로 저장되어 있지만 범주처럼 다루는 열이 있다면 이유와 함께 쓰세요. 이 분석 질문(시간대별 평균 대여량)에서 `hour`와 `rented_count`는 각각 어느 쪽으로 다룹니까?

> **교수용 예시**
> - 범주형 `season`, `holiday`, `functioning_day`, `date`(날짜를 구분하는 값)
> - 연속형 `rented_count`, `temperature`, `humidity`, `wind_speed`, `visibility`, `dew_point`, `solar_radiation`, `rainfall`, `snowfall`
> - `hour`는 0~23의 숫자지만 시간대를 서로 비교하는 **기준**으로 쓰므로 범주처럼 다룹니다. 평균을 낼 대상은 연속형인 `rented_count`입니다.

In [ ]:
print(bike.dtypes)

### 1.3 Series 만들기

`bike`의 **첫 24행**은 첫째 날의 24시간 기록입니다. 이 24행에서 대여량만 꺼내 `day1`로 저장하세요. 열 하나를 꺼낸 결과의 **타입**을 확인하고, 레이블 인덱스, 값의 개수, 형태, 자료형을 확인하세요. 레이블은 어떤 값입니까?

> **확인** 타입은 Series, 값의 개수는 24, 형태는 (24,), 자료형은 int64입니다. 레이블은 0~23으로, 이 날은 행 번호가 곧 시간(`hour`)과 같습니다.

In [ ]:
day1 = bike.loc[0:23, "rented_count"]
print(type(day1))
print("레이블:", day1.index.tolist())
print("값의 개수:", day1.size)
print("형태:", day1.shape)
print("자료형:", day1.dtype)

### 1.4 레이블과 절대 위치로 값 선택

`day1`에서 다음을 구하세요.

1. 절대 위치 5의 값과 레이블 18의 값
2. 절대 위치 6~8 범위와 레이블 6~8 범위에서 각각 **값이 몇 개**인지

그다음 둘째 날의 대여량 `day2`(행 24~47)를 만들고, **레이블 30**의 값과 **절대 위치 6**의 값을 비교하세요. 두 값이 같은 시간의 값입니까? 왜 이런 결과가 나옵니까?

> **확인** 위치 5의 값은 100건, 레이블 18의 값은 862건입니다. 위치 6~8 범위는 끝을 제외해 3개, 레이블 6~8 범위는 끝을 포함해 4개입니다.
> `day2`의 레이블은 24~47이라서 레이블 30(=70건)과 절대 위치 6(=70건)은 **같은 값**입니다. 둘째 날부터는 레이블과 위치가 서로 다르므로 두 방식을 구분해야 합니다.

In [ ]:
print(day1.iloc[5], day1.loc[18])
print(len(day1.iloc[6:9]), len(day1.loc[6:9]))

day2 = bike.loc[24:47, "rented_count"]
print(day2.loc[30], day2.iloc[6])

### 1.5 Series의 조건 추출, 산술, 통계

`day1`에서 다음을 구하세요.

1. 대여량이 **500 이상**인 시간(레이블)과 값
2. 대여량이 **200 이상 500 미만**인 시간(레이블)
3. 평균, 중앙값, 최댓값, 그리고 최댓값이 나온 **시간**
4. 대여량을 **10% 늘린** 새로운 Series. 원래의 `day1`은 바뀌었습니까?

> **확인** 500건 이상인 시간 [8, 17, 18, 19], 200건 이상 500건 미만인 시간 [0, 1, 7, 9, 10, 11, 12, 13, 14, 15, 16, 20, 21, 22, 23]입니다. 평균 397.5건, 중앙값 415.5건, 최댓값 930건(시간 8시)입니다. 10% 늘린 Series를 만들어도 `day1`은 바뀌지 않습니다.

In [ ]:
print(day1[day1 >= 500])
print(day1[(day1 >= 200) & (day1 < 500)].index.tolist())

print("평균:", round(day1.mean(), 1))
print("중앙값:", day1.median())
print("최댓값:", day1.max(), "→", day1.idxmax(), "시")

increased = day1 * 1.1
print(day1.loc[8], round(increased.loc[8], 1))

### 1.6 대입과 copy()

`day1`의 앞 세 값(위치 0~2)으로 새 Series `a`를 만들되, 원본에 영향이 없도록 복사본으로 만드세요. 그다음 `b = a`와 `c = a.copy()`를 만들고, `b`의 레이블 0 값을 0으로 바꾼 뒤 `a`, `b`, `c`의 레이블 0 값을 각각 출력하세요. 무엇이 바뀌었습니까? 분석에서 원본을 보존하는 방법을 한 줄로 쓰세요.

> **확인** `b`를 바꾸자 `a`도 함께 0으로 바뀌고, `c`는 원래 값을 유지합니다. `b = a`는 복사가 아니라 같은 객체를 가리키기 때문입니다. 원본 데이터는 그대로 두고 `copy()`한 복사본이나 정제 결과를 새 이름에 저장해 작업합니다.

In [ ]:
a = day1.iloc[0:3].copy()
b = a
c = a.copy()

b.loc[0] = 0
print("a:", a.loc[0], "| b:", b.loc[0], "| c:", c.loc[0])

### 1.7 데이터프레임 확인

`bike`에는 무엇이 저장되어 있습니까? 파일 이름입니까, 표입니까? 자료형을 코드로 확인하고, 이 데이터에서 **관측치(행) 하나**와 **변수(열) 하나**가 각각 무엇을 나타내는지 설명하세요.

**확인값** `pandas`의 `DataFrame`입니다.

> 관측치(행) 하나는 **날짜와 시간으로 정해지는 한 시간 동안의 대여 건수와 날씨 기록**이고, 변수(열) 하나는 그 기록의 **한 가지 속성**(대여량, 기온 등)입니다.

In [ ]:
print(type(bike))

### 1.8 데이터 미리보기

데이터의 처음 5행, 마지막 3행, 무작위 표본 3행(`random_state=10`으로 고정)을 미리 보세요. 세 가지를 **모두** 확인하는 이유를 설명하세요.

**확인값** 마지막 행의 날짜는 `30/11/2018`, 시간은 `23`입니다.

> 처음과 끝만 보면 중간의 문제를 놓칠 수 있어 무작위 행도 함께 봅니다. 행이 날짜·시간 순서로 정렬되어 있다는 것도 알 수 있습니다.

In [ ]:
display(bike.head())
display(bike.tail(3))
display(bike.sample(3, random_state=10))

### 1.9 데이터 크기와 변수 자료형

관측치(행)와 변수(열)의 수, 변수 이름, 변수별 자료형을 확인하세요. 수치형 변수 세 가지와 문자열(범주형) 변수 세 가지를 적으세요.

**확인값** 행과 열은 `(8760, 14)`입니다.

> 수치형 `rented_count`, `hour`, `temperature` / 문자열 `date`, `season`, `holiday`

In [ ]:
print(bike.shape)
print(bike.columns.tolist())
print(bike.dtypes)

### 1.10 info()로 데이터 구조 확인

`info()`를 실행하고 `Non-Null Count`와 `Dtype`을 읽으세요. 비어 있는 값이 있는 열이 있습니까? `date`는 어떤 자료형이며, **이 질문에서** 날짜를 날짜형으로 바꿀 필요가 있습니까?

> 모든 열이 8760으로 같아 비어 있는 값이 없습니다. `date`는 문자열이지만 이 질문은 `hour`로 나누므로 날짜형으로 바꿀 필요가 없습니다.

In [ ]:
bike.info()

### 1.11 열 하나 꺼내기와 고유값

`bike`에서 열 하나(`season`)를 꺼내 **타입**을 확인하세요. 그다음 `season`, `holiday`, `functioning_day` 세 열이 각각 가진 **고유한 값**을 확인하세요. 계절은 몇 종류입니까?

> **확인** 열 하나를 꺼내면 Series입니다. `season`은 ['Winter', 'Spring', 'Summer', 'Autumn']의 4종류, `holiday`는 ['No Holiday', 'Holiday'], `functioning_day`는 ['Yes', 'No']입니다.

In [ ]:
print(type(bike["season"]))
print(bike["season"].unique())
print(bike["holiday"].unique())
print(bike["functioning_day"].unique())

### 1.12 행과 열을 함께 선택

`bike`의 행 0~2에서 `date`, `hour`, `rented_count` 세 열을 ① **열 이름**으로 ② **열 위치**로 각각 선택하세요. 열 위치로 고르려면 열 순서를 알아야 합니다. 두 결과에서 **행의 개수**와 **열의 순서**는 같습니까?

> **확인** 두 결과 모두 행은 3개입니다(`loc`는 끝을 포함하고 `iloc`는 끝을 제외하므로 범위를 0~2, 0~3으로 쓴 것입니다). 이름으로 고르면 `date`, `hour`, `rented_count` 순서이고, 원래 열 순서가 `date`, `rented_count`, `hour`여서 위치 0~2로 고르면 `date`, `rented_count`, `hour` 순서가 됩니다.

In [ ]:
print(bike.loc[0:2, ["date", "hour", "rented_count"]])
print(bike.iloc[0:3, 0:3])

### 1.13 변수 선택과 필터링

질문에 필요한 변수 두 개만 선택해 새 변수로 저장하고 앞부분을 확인하세요. 그다음 **18시 관측치**만 필터링해 몇 건인지 세세요.

**확인값** 18시 기록은 365건입니다. (1년이 365일이므로 하루에 한 건씩)

In [ ]:
selected = bike[["hour", "rented_count"]]
display(selected.head())

evening = bike[bike["hour"] == 18]
print("18시 기록 수:", len(evening))

### 1.14 복합 조건 필터링

**18시이고 기온이 25도 이상인** 관측치가 몇 건인지 구하세요. 조건 두 개를 함께 쓸 때 코드에서 주의할 점을 한 줄로 쓰세요.

**확인값** 92건입니다.

> 각 조건을 괄호로 묶고 `&`로 연결합니다.

In [ ]:
hot_evening = bike[
    (bike["hour"] == 18)
    & (bike["temperature"] >= 25)
]
print("18시이고 25도 이상인 기록 수:", len(hot_evening))

### 1.15 새로운 열 만들기

풍속(`wind_speed`, m/s)을 시속(km/h)으로 환산한 열 `wind_kmh`를 만드세요. 1 m/s는 3.6 km/h입니다. 원본 `bike`를 보존하기 위해 복사본 `bike2`에서 작업하고, 소수 첫째 자리까지 반올림하세요. 풍속이 가장 센 관측치 3개의 풍속과 환산값을 확인하세요.

> **확인** 풍속이 가장 센 관측치의 풍속은 7.4 m/s이고 시속으로 26.6 km/h입니다. 새 열은 같은 행의 값끼리 계산되어 만들어졌고 `bike`에는 `wind_kmh`가 없습니다.

In [ ]:
bike2 = bike.copy()
bike2["wind_kmh"] = (bike2["wind_speed"] * 3.6).round(1)

bike2.sort_values("wind_speed", ascending=False)[["date", "hour", "wind_speed", "wind_kmh"]].head(3)

### 1.16 정렬로 상위 관측치 확인

대여량이 **가장 큰** 관측치 5개를 날짜, 시간, 대여량, 계절과 함께 확인하세요. 어떤 공통점이 있습니까?

**확인값** 가장 큰 값은 3556입니다.

> 5개 모두 **여름(6월)의 18시**입니다. 큰 값이 임의의 오류가 아니라 특정 조건(계절과 시간)에 집중되어 있습니다.

In [ ]:
bike.sort_values("rented_count", ascending=False)[["date", "hour", "rented_count", "season"]].head(5)

### 1.17 열 삭제

`bike2`에서 `dew_point`와 `visibility` 열을 삭제한 `slim`을 만들고, `bike2`와 `slim`의 열 개수를 비교하세요. 삭제한 뒤에도 `bike2`의 열이 그대로인 이유는 무엇입니까?

> **확인** `bike2`는 15개(원본 14개 + `wind_kmh`), `slim`은 13개입니다. `drop()`은 삭제한 결과를 새 DataFrame으로 돌려줄 뿐 원본을 바꾸지 않기 때문입니다.

In [ ]:
slim = bike2.drop(columns=["dew_point", "visibility"])
print("bike2 열 개수:", bike2.shape[1])
print("slim 열 개수:", slim.shape[1])

### 1.18 결측값 확인

변수별 결측값의 개수를 구하고, 데이터 전체의 합계도 구하세요.

**확인값** 합계는 0입니다.

> 모든 열이 0이고 전체 합계도 0입니다. **결측이 없다는 것도 확인한 결과**입니다.

In [ ]:
print(bike.isna().sum())
print("전체:", bike.isna().sum().sum())

### 1.19 중복값 확인

중복된 행이 몇 개인지 구하고, 있다면 중복된 행을 **모두** 표시하는 방법으로 확인하세요.

**확인값** 0건입니다. 겹친 행을 보여 주는 코드는 빈 표를 돌려줍니다.

> 0건이며, 겹친 행을 보여 주는 코드(`keep=False`)도 빈 표를 돌려줍니다.

In [ ]:
print("중복 행:", bike.duplicated().sum())
bike[bike.duplicated(keep=False)]

### 1.20 비정상 값 탐색

결측값도 중복값도 없습니다. 그런데 이 데이터를 그대로 분석에 사용해도 될까요? 대여량이 가장 **작은** 관측치부터 확인하세요. 비정상적이거나 설명이 필요한 값이 있습니까? 있다면 **몇 건인지**, **어떤 공통점이 있는지**를 코드로 확인하세요. (힌트 다른 변수와 함께 보면 단서가 있을 수 있습니다.)

**확인값** 대여량이 0인 행은 295건입니다. 공통점을 찾는 열은 하나입니다.

> 대여량이 0인 295행은 **모두 운영하지 않은 날**(`functioning_day`가 `No`)이고 13일에 걸쳐 있습니다. 시간대와 상관없이 하루 종일 0이므로 "빌릴 사람이 없었다"가 아니라 **운영을 하지 않아 0으로 기록된 것**입니다.

In [ ]:
low = bike.sort_values("rented_count")
display(low[["date", "hour", "rented_count", "functioning_day"]].head(5))

zero_rows = bike[bike["rented_count"] == 0]
print("대여량이 0인 행:", len(zero_rows))
print(zero_rows["functioning_day"].value_counts())

### 1.21 데이터 전처리 방법 결정

이 관측치들의 전처리 방법을 두 가지로 **평균 대여량을 계산해 비교**한 뒤 결정하세요.

- 방법 A 모든 관측치를 그대로 사용
- 방법 B 문제가 되는 관측치를 제외하고 사용 (제외한 데이터를 `operating`이라는 이름으로 저장하세요)

**무엇을 제외했고 왜 그렇게 했는지** 전처리 기준을 글로 쓰세요. 이후 분석은 선택한 데이터로 합니다.

**확인값** 방법 A 704.6 / 방법 B 729.2 / 제외한 행 295

> 방법 B를 선택했습니다.
> - 제외한 관측치 `functioning_day`가 `No`인 295행
> - 제외한 이유 운영하지 않은 날의 0건은 대여 수요가 아니어서, 포함하면 수요를 실제보다 낮게 계산하기 때문

In [ ]:
all_mean = bike["rented_count"].mean()

operating = bike[bike["functioning_day"] == "Yes"]
op_mean = operating["rented_count"].mean()

print("방법 A 전체 평균:", round(all_mean, 1))
print("방법 B 운영일만 평균:", round(op_mean, 1))
print("제외한 행:", len(bike) - len(operating))

### 1.22 이상값 탐지 기준

1.16에서 대여량이 매우 큰 관측치를 확인했습니다. 이 값들을 **이상값으로 판정할 기준**을 두 가지로 세워 비교하세요. `operating`의 대여량으로 계산합니다.

- 기준 ① 3사분위수 + 1.5 × (3사분위수 − 1사분위수)
- 기준 ② 평균 + 2 × 표준편차

각 기준의 **상한값**과, 상한을 **초과하는 관측치의 수**를 구하세요. 초과한 관측치가 어느 시간대와 계절에 집중되어 있는지도 확인한 뒤, **이 관측치들을 제외해야 하는지** 판단하세요.

**확인값** 기준 ① 상한 2389.0, 초과 152건 / 기준 ② 상한 2013.9, 초과 470건

> 두 기준이 걸러내는 건수가 다릅니다(152건, 470건). 기준을 넘는 기록은 **18~19시 여름·봄·가을**에 몰려 있습니다. 오류가 아니라 **실제로 대여가 많은 시간대의 값**이므로 **제외하지 않습니다.** 이론의 카페 45분(다른 값과 동떨어진 한 건)과 달리, 이 값들은 이유를 설명할 수 있기 때문입니다.

In [ ]:
r = operating["rented_count"]
d = r.describe()
iqr_limit = d["75%"] + 1.5 * (d["75%"] - d["25%"])
sd_limit = d["mean"] + 2 * d["std"]

print("기준 1 상한:", iqr_limit, "| 초과:", (r > iqr_limit).sum(), "건")
print("기준 2 상한:", round(sd_limit, 1), "| 초과:", (r > sd_limit).sum(), "건")

flagged = operating[r > iqr_limit]
print(flagged["hour"].value_counts().head(3))
print(flagged["season"].value_counts())

### 1.23 기술통계 요약

`operating`의 대여량을 요약하세요.

1. 개수, 평균, 표준편차, 사분위수, 최댓값을 한 번에 확인하세요.
2. 평균과 중앙값을 비교하고, 차이가 말해 주는 것을 쓰세요.
3. 계절과 공휴일 여부의 **빈도와 비율(%)**을 구하세요. 비율이 한쪽으로 치우친 변수가 있다면, 그것이 그룹 비교에 어떤 영향을 줄지 생각해 보세요.

**확인값** 평균 729.2, 중앙값 542.0, 75% 1084, 최댓값 3556 / 공휴일 `No Holiday` 95.2%, `Holiday` 4.8%

> 평균(729.2)이 중앙값(542.0)보다 큽니다. 값의 75%가 1084건 이하인데 최댓값이 3556건이라, 대여가 많은 일부 기록이 평균을 끌어올렸습니다. 계절은 비슷한 비율이지만 공휴일은 4.8%(408건)뿐이어서, 공휴일 평균은 **건수가 적어 불안정**할 수 있습니다.

In [ ]:
r = operating["rented_count"]
display(r.describe().round(1))
print("평균:", round(r.mean(), 1), "| 중앙값:", r.median())

for col in ["season", "holiday"]:
    print(operating[col].value_counts())
    print((operating[col].value_counts(normalize=True) * 100).round(1))
    print()

### 1.24 그룹별 집계와 분석 질문 답하기

처음 질문에 답합니다.

1. 시간대별로 **관측치 수, 평균, 중앙값**을 집계하세요. 표본 크기가 충분한지 `count`로 확인하세요.
2. 평균이 가장 큰 시간대를 **코드로** 찾고, 상위 3개와 하위 3개 시간대를 확인하세요.
3. 1.21의 전처리 결정이 결론을 바꾸는지 확인하세요(민감도 확인). `operating`과 `bike`(전체)로 같은 집계를 해 **나란히** 놓고 비교하세요. 가장 많은 시간대가 달라집니까? 값은 얼마나 달라집니까?
4. 질문에 **한 문장**으로 답하세요.

**확인값** 많은 3곳 18시 1554.0, 19시 1235.8, 17시 1177.2 / 가장 적은 곳 4시 137.5 / 시간대마다 352~353건

> 가장 많은 시간대는 **18시(평균 1554.0건)**로 4시(평균 137.5건)의 약 11.3배입니다. 비운영일을 포함해도 가장 많은 시간대는 18시로 **같지만**, 평균은 1502.9건에서 1554.0건으로 달라집니다. 순위는 그대로여도 "얼마나 많은가"는 처리 방법에 따라 달라집니다.

In [ ]:
hourly = (
    operating
    .groupby("hour")["rented_count"]
    .agg(["count", "mean", "median"])
    .round(1)
)
display(hourly)

best_hour = hourly["mean"].idxmax()
print("가장 많은 시간대:", best_hour, "시,", hourly.loc[best_hour, "mean"], "건")
print(hourly["mean"].sort_values(ascending=False).head(3))
print(hourly["mean"].sort_values().head(3))

hourly_all = bike.groupby("hour")["rented_count"].agg(["count", "mean"]).round(1)
compare = pd.DataFrame({"운영일만": hourly["mean"], "전체": hourly_all["mean"]})
compare["차이"] = (compare["운영일만"] - compare["전체"]).round(1)
compare.sort_values("차이", ascending=False).head(3)

# 2. 문제 정의에서 시작하는 분석

이번에는 **모호한 문제**가 주어집니다. 문제를 분석 질문으로 정의하고, **EDA로 데이터를 점검하며 분석 방법을 설계**합니다.

> 운영 담당자 "날씨가 자전거 이용에 영향을 주는 것 같습니다. 확인해 주세요."

데이터는 1에서 정리한 `operating`을 그대로 씁니다.

### 2.1 분석 질문 정의

위 요청을 **분석 질문**으로 바꾸세요. 질문의 다섯 요소(대상, 측정값, 기준, 계산, 범위)를 모두 쓰고, 날씨 변수 중 **어떤 변수를 기준으로** 비교할지 고르세요. 질문이 "영향을 준다(인과)"를 묻고 있지는 않은지 점검하세요.

**확인값** 날씨 열은 `temperature`, `humidity`, `wind_speed`, `rainfall`, `snowfall` 등이 있습니다. `rainfall`은 0보다 큰지, `temperature`는 구간으로 나눌 수 있습니다.

> **교수용 예시** 비(`rainfall` > 0) 여부로 나눠 평균 대여량을 비교합니다.
> - 대상 한 시간 동안의 대여 기록 / 측정값 `rented_count` / 기준 비 여부 / 계산 평균 / 범위 운영한 날의 1년
> - 질문 비가 오는 시간의 평균 대여량은 비가 오지 않는 시간과 어떻게 다른가?
> - "영향을 준다"를 "차이가 있는가"로 낮춰서 원인 판단을 피했습니다.

### 2.2 데이터 적합성 점검

분석하기 전에 데이터가 질문에 답하기에 적합한지 **EDA로 점검**하세요. 내가 고른 기준으로 나눈 **그룹별 관측치 수**를 확인하고, 한쪽이 너무 적지는 않은지 보세요.

**확인값** 기준을 `rainfall > 0`으로 골랐다면 전체 8,465건 중 비가 오는 시간은 516건입니다.

> 비 오는 시간이 516건(6.1%)이고 나머지가 7,949건입니다. 한쪽이 훨씬 적습니다. 516건은 비교하기에는 충분하지만 **그룹 크기가 매우 다르다**는 점을 기억해야 합니다.

In [ ]:
rain = operating[operating["rainfall"] > 0]
dry = operating[operating["rainfall"] == 0]
print("비 오는 시간:", len(rain), "/ 오지 않는 시간:", len(dry))

### 2.3 분석 방법 설계

바로 비교하기 전에, **단순 비교가 타당한지** 데이터로 확인하세요. 비가 오는 시간이 특정 계절에 편중되어 있다면, 계절이 비와 대여량 **양쪽에 영향을 주는 교란 요인**이 되어 비의 효과가 왜곡될 수 있습니다.

1. 계절마다 비 오는 시간의 **비율**을 구하세요.
2. 결과를 보고, **교란 요인을 어떻게 통제하며 비교할지** 분석 방법을 쓰세요.

**확인값** 계절별 비 오는 시간 비율(%) Autumn 4.5, Spring 8.8, Summer 8.9, Winter 1.9

> 비 오는 시간의 비율이 계절마다 달라서(Autumn 4.5%, Spring 8.8%, Summer 8.9%, Winter 1.9%) 전체를 한꺼번에 비교하면 계절의 영향이 섞입니다. 그래서 **계절을 고정(층화)한 채로 비 오는 시간과 오지 않는 시간을 비교**하는 방법으로 설계합니다.

In [ ]:
rain_season = rain["season"].value_counts()
all_season = operating["season"].value_counts()
(rain_season / all_season * 100).round(1)

### 2.4 비교 분석 실행

11.7에서 설계한 방법으로 비교하세요. 먼저 **단순 비교**(계절 구분 없음)를 하고, 그다음 **계절을 고정한 비교**를 하세요. 두 결과를 나란히 놓고, 비 오는 쪽 평균이 낮은 것이 **모든 계절에서** 나타나는지 확인하세요. 그룹의 **건수**도 반드시 함께 읽으세요.

**확인값** 단순 비교 False 평균 765.6, True 평균 167.3

- `Spring` 비 안 옴 1,969건 평균 809.7 / 비 옴 191건 평균 91.8
- `Summer` 비 안 옴 2,011건 평균 1110.1 / 비 옴 197건 평균 257.9
- `Autumn` 비 안 옴 1,850건 평균 959.8 / 비 옴 87건 평균 165.3
- `Winter` 비 안 옴 2,119건 평균 228.2 / 비 옴 41건 평균 87.4

In [ ]:
print("단순 비교")
print("비 옴:", round(rain["rented_count"].mean(), 1), "| 비 안 옴:", round(dry["rented_count"].mean(), 1))

by_season = pd.DataFrame({
    "비 옴 건수": rain.groupby("season")["rented_count"].count(),
    "비 옴 평균": rain.groupby("season")["rented_count"].mean(),
    "비 안 옴 평균": dry.groupby("season")["rented_count"].mean(),
}).round(1)
by_season

### 2.5 결과 해석과 한계

세 가지로 나누어 쓰세요. **확인한 사실**(숫자 포함), **가능한 해석**(인과를 단정하지 않기), **한계**(이 결과가 말해 주지 못하는 것). 한계에는 **표본이 작은 그룹**과 **통제하지 못한 다른 조건**을 포함하세요.

> **사실** 네 계절 모두 비 오는 시간의 평균 대여량이 오지 않는 시간보다 낮다. 여름은 1110.1건에서 257.9건으로 줄었다.
> **해석** 비가 대여를 줄이는 것과 관련이 있을 가능성이 있다.
> **한계** 겨울은 비 오는 시간이 41건뿐이라 평균이 불안정하다. 시간대나 기온처럼 함께 달라지는 다른 조건은 따로 통제하지 않았으므로 비가 **원인**이라고 단정할 수 없다.

# 3. 탐색에서 시작하는 문제 정의

이번에는 **주어진 분석 질문이 없습니다.** 탐색적 데이터 분석을 먼저 수행하고, 탐색에서 드러난 패턴이 **아직 설명되지 않은 채 남긴 의문**에서 분석 질문을 도출합니다. 탐색은 질문을 만드는 데까지이고, 질문에 답하는 분석은 그다음 단계입니다.

데이터는 계속 `operating`을 씁니다.

### 3.1 계절별 집계

`season`별로 관측치 수, 평균, 중앙값을 구하세요. 평균 대여량이 가장 큰 계절과 가장 작은 계절은 무엇이며, 몇 배 차이가 납니까?

> **관찰** 평균이 가장 큰 계절은 여름(1034.1건), 가장 작은 계절은 겨울(225.5건)로 약 4.6배 차이입니다. 계절에 따라 대여량의 수준이 크게 다릅니다.

In [ ]:
operating.groupby("season")["rented_count"].agg(["count", "mean", "median"]).round(1)

### 3.2 공휴일 여부별 집계

`holiday`별로 같은 집계를 하세요. 어느 쪽 평균이 더 높습니까? 공휴일 기록은 몇 건입니까?

> **관찰** 공휴일 평균은 529.2건으로 공휴일이 아닌 때(739.3건)보다 210.1건 낮습니다. 공휴일 기록은 408건뿐입니다.

In [ ]:
operating.groupby("holiday")["rented_count"].agg(["count", "mean", "median"]).round(1)

### 3.3 공휴일 기록의 계절 구성

3.2의 차이를 그대로 믿어도 되는지 점검합니다. 공휴일 기록만 따로 만들어 **계절별 건수와 비율(%)**을 구하고, `operating` 전체의 계절별 비율과 나란히 놓으세요. 공휴일이 어느 계절에 몰려 있습니까?

> **관찰** 공휴일 기록의 47.1%(192건)가 겨울에 있습니다. 전체에서 겨울의 비율은 25.5%입니다. 3.1에서 겨울은 대여량이 가장 낮은 계절이었습니다. 그렇다면 3.2의 차이는 공휴일 때문일 수도 있고, 공휴일이 겨울에 몰려 있기 때문일 수도 있습니다. **이 탐색만으로는 둘을 구분할 수 없습니다.**

In [ ]:
holiday_rows = operating[operating["holiday"] == "Holiday"]

share = pd.DataFrame({
    "공휴일 건수": holiday_rows["season"].value_counts(),
    "공휴일 비율(%)": holiday_rows["season"].value_counts(normalize=True) * 100,
    "전체 비율(%)": operating["season"].value_counts(normalize=True) * 100,
}).round(1)
share

### 3.4 발견한 패턴과 남은 의문 정리

3.1에서 3.3까지의 결과에서 **발견한 패턴 세 가지**를 근거 수치와 함께 쓰세요. 그다음 **탐색으로는 설명되지 않은 채 남은 의문**을 한 가지 이상 쓰세요. 아직 질문 형태로 만들지는 않습니다.

> **교수용 예시**
> - 패턴 1 평균 대여량은 여름(1034.1건)이 겨울(225.5건)의 약 4.6배이다.
> - 패턴 2 공휴일 평균(529.2건)이 공휴일이 아닌 때(739.3건)보다 낮다.
> - 패턴 3 공휴일 기록의 47.1%가 겨울에 있다.
> - **남은 의문** 공휴일의 평균이 낮은 것이 공휴일 때문인지, 겨울에 몰려 있기 때문인지 알 수 없다.

### 3.5 탐색 결과에서 분석 질문 도출

3.4에서 남은 의문을 **분석 질문**으로 바꾸세요. 다섯 요소(대상, 측정값, 기준, 계산, 범위)를 모두 쓰고, 다음을 점검하세요.

1. 이 데이터에 필요한 변수가 있습니까?
2. 계산하면 답이 나옵니까? ("달라 보인다", "원인이다"가 아니라)
3. 비교하는 그룹의 **표본 크기가 충분**합니까?
4. 이 질문의 답이 **지금까지의 탐색 결과에 이미 나와 있지는 않습니까?** 이미 나와 있다면 새로운 분석이 필요하지 않은 질문입니다.

> **의문에서 도출되는 질문 후보**
>
> | 질문 | 점검 | 판단 |
> |:---|:---|:---|
> | 공휴일 평균이 낮은 **이유는** 무엇인가? | 원인을 묻는 질문이라 이 자료로 계산할 수 없음 | 고쳐야 함 |
> | 공휴일 여부별 평균 대여량은 다른가? | 3.2에서 이미 답이 나와 있음 | 새 분석이 필요 없음 |
> | **계절을 같게 놓고 비교하면 공휴일과 공휴일이 아닌 때의 평균 대여량은 어떻게 다른가?** | 변수가 있고, 계산하면 답이 나오며, 탐색에서는 구분하지 못한 부분임 | 적절함 |
>
> **교수용 예시** 세 번째 질문을 고릅니다.
> - 대상 한 시간 동안의 대여 기록 / 측정값 `rented_count` / 기준 공휴일 여부(계절별로 나누어) / 계산 평균 / 범위 이 자료의 1년
> - 점검 필요한 변수(`holiday`, `season`, `rented_count`)가 있습니다. 공휴일 기록은 계절마다 48~192건이라 **적지만 비교는 가능**하고, 결론의 범위를 이 자료로 한정합니다. 3.2~3.3에서는 계절을 구분한 비교를 하지 않았으므로 답이 아직 나와 있지 않습니다.
> - 이 질문은 3.3에서 발견한 **교란 가능성**에서 나왔습니다. 탐색이 질문을 만들고, 답은 다음 분석에서 얻습니다.
> - 학생이 다른 질문을 골라도 네 가지 점검을 통과하는지로 확인합니다.

### 3.6 도출한 분석 질문에 답하기

3.5에서 정의한 분석 질문에 답하는 코드를 작성하세요. 지금까지의 탐색에서 하지 않은 **새로운 비교**여야 합니다. 결과를 해석할 때 그룹의 **표본 크기**를 함께 확인하고, 3.2의 결과와 **비교**해서 질문에 한 문장으로 답하세요.

> **교수용 예시 (계절을 같게 놓고 비교한 질문일 때)**
> 계절을 구분하지 않으면 공휴일이 210.1건 낮았습니다. 계절별로 나누어 비교하면 여름(1022.1건, 1034.3건)과 가을(948.1건, 922.9건)은 차이가 거의 없고, 봄(-114.8건)과 겨울(-75.7건)에서만 공휴일이 낮습니다. 즉 3.2의 차이는 상당 부분 **공휴일이 겨울에 몰려 있는 구성 때문**이었고, 공휴일 자체의 영향은 계절에 따라 다르거나 작게 나타납니다.
> **한계** 공휴일 기록은 계절마다 48~192건(2~8일)이라 적고, 시간대와 기온은 통제하지 않았습니다. 공휴일이 원인이라고 단정할 수 없습니다.

In [ ]:
normal_rows = operating[operating["holiday"] == "No Holiday"]

by_season = pd.DataFrame({
    "공휴일 건수": holiday_rows.groupby("season")["rented_count"].count(),
    "공휴일 평균": holiday_rows.groupby("season")["rented_count"].mean(),
    "비공휴일 건수": normal_rows.groupby("season")["rented_count"].count(),
    "비공휴일 평균": normal_rows.groupby("season")["rented_count"].mean(),
}).round(1)
by_season["차이"] = (by_season["공휴일 평균"] - by_season["비공휴일 평균"]).round(1)
by_season

### 3.7 결과 종합과 제안

처음의 모호한 문제("자전거가 언제, 어떤 때 많이 쓰이는지 알고 싶다")로 돌아갑니다. 지금까지의 분석 결과를 종합해 답하세요.

1. 이 데이터로 **확인할 수 있는 사실(인사이트)** 세 가지를 쓰세요. 각각 근거 수치를 함께 쓰세요.
2. 이 데이터만으로는 **확인할 수 없는 것** 두 가지를 쓰세요. 왜 확인할 수 없는지도 설명하세요.
3. 운영 담당자에게 **한 가지를 제안**한다면 무엇을 제안하겠습니까? **근거 수치**와, 그 근거의 **한계**를 함께 쓰세요.
4. 제안의 근거를 강화하려면 **어떤 추가 데이터가 필요**합니까?

> **교수용 모범 예시**
>
> **1 확인할 수 있는 사실**
> 1. 시간대에 따라 대여량이 크게 다릅니다. 18시 평균 1554.0건, 4시 평균 137.5건으로 약 11.3배 차이가 납니다.
> 2. 비 오는 시간의 평균 대여량이 오지 않는 시간보다 낮고, 이 차이는 네 계절 모두에서 나타납니다. (여름 1110.1건에서 257.9건)
> 3. 공휴일의 평균 대여량(529.2건)은 공휴일이 아닌 때(739.3건)보다 낮아 보이지만, 계절을 같게 놓고 보면 여름과 가을은 차이가 거의 없습니다. 공휴일이 겨울에 몰려 있었기 때문입니다.
>
> **2 확인할 수 없는 것**
> 1. **왜** 18시에 많은지는 알 수 없습니다. 이용 목적(출퇴근, 여가)이 기록되어 있지 않습니다.
> 2. **어느 대여소**에서 부족한지는 알 수 없습니다. 대여소 위치가 없고 서울 전체의 시간별 합계만 있습니다.
>
> **3 제안** 17~19시에 자전거 재배치와 정비 인력을 집중합니다. 근거는 18시 평균 1554.0건이 새벽의 약 11.3배라는 점입니다. 다만 이 근거는 **서울 전체의 합계**라서 실제로 부족한 대여소가 어디인지는 보여 주지 못합니다.
>
> **4 추가로 필요한 데이터** 대여소별 대여·반납 건수, 대여소별 자전거 보유 대수, 이용 목적